goal = combine cbioportal data + extra endpoints

In [1]:
import pandas as pd

In [2]:
data_path = '../../data'

In [3]:
# load cbioportal data

df_cbio = pd.read_csv(f'{data_path}/clinical_data/brca_tcga_gdc_clinical_data.tsv', sep = '\t')
df_cbio = df_cbio[["Sample ID", "Patient ID"]]
columns={
    "Sample ID" : "case_id",
    "Patient ID" : "patient_id"
    }
df_cbio = df_cbio.rename(columns=columns)
df_cbio.head()

,case_id,patient_id
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU
1,TCGA-3C-AALI-01,TCGA-3C-AALI
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ
3,TCGA-3C-AALK-01,TCGA-3C-AALK
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK


In [4]:
# load extra endpoints data

df_extr = pd.read_csv(f'{data_path}/clinical_data/extra_endpoints.csv', sep=';', index_col=0)
df_extr = df_extr[["bcr_patient_barcode", "DSS.time.cr", "DSS_cr"]]
columns={
    "bcr_patient_barcode" : "patient_id",
    "DSS.time.cr" : "dss_time",
    "DSS_cr" : "dss_censorship"
    }
df_extr = df_extr.rename(columns=columns)
df_extr.head()

,patient_id,dss_time,dss_censorship
1,TCGA-OR-A5J1,1355.0,1.0
2,TCGA-OR-A5J2,1677.0,1.0
3,TCGA-OR-A5J3,2091.0,0.0
4,TCGA-OR-A5J4,423.0,1.0
5,TCGA-OR-A5J5,365.0,1.0


In [5]:
# merge dataframes

joined = pd.merge(df_cbio, df_extr, on=['patient_id'])
joined.head()

,case_id,patient_id,dss_time,dss_censorship
0,TCGA-3C-AAAU-01,TCGA-3C-AAAU,4047.0,0.0
1,TCGA-3C-AALI-01,TCGA-3C-AALI,4005.0,0.0
2,TCGA-3C-AALJ-01,TCGA-3C-AALJ,1474.0,0.0
3,TCGA-3C-AALK-01,TCGA-3C-AALK,1448.0,0.0
4,TCGA-4H-AAAK-01,TCGA-4H-AAAK,348.0,0.0


In [6]:
print("Rows with nan survival time: ", joined['dss_time'].isna().sum())
# deleting nan rows
joined = joined[joined['dss_time'].notna()]

Rows with nan survival time:  1


In [7]:
print("Rows with nan censorship: ", joined['dss_censorship'].isna().sum())
# deleting nan rows
joined = joined[joined['dss_censorship'].notna()]

Rows with nan censorship:  20


In [8]:
joined['dss_censorship'].value_counts(dropna=False)

dss_censorship
0.0    949
1.0     85
2.0     49
Name: count, dtype: int64

In [9]:
# change censorship signs to be the same as our data
joined['dss_censorship'] = joined['dss_censorship'].map({0: 1, 1: 0, 2: 1})

In [10]:
# save to file
joined.to_csv(f"{data_path}/clinical_data/brca_clinical.csv")